In [7]:
import pandas as pd
import requests
import time
import os
from geopy.geocoders import Nominatim


In [9]:
# 1. Load the dataset
df = pd.read_csv('../data/processed/clean_crop_yield_data.csv')
unique_locations = df[['State', 'District']].drop_duplicates().reset_index(drop=True)

geolocator = Nominatim(user_agent="agroplan_x_weather_bot")
weather_records = []

def get_coordinates(state, district):
    query = f"{district}, {state}, India"
    try:
        location = geolocator.geocode(query, timeout=10)
        if location:
            return location.latitude, location.longitude
    except Exception as e:
        print(f"Geocode error for {district}: {e}")
    return None, None

def get_nasa_weather(lat, lon, start="2014", end="2025"):
    url = f"https://power.larc.nasa.gov/api/temporal/monthly/point?parameters=T2M,PRECTOTCORR&community=AG&longitude={lon}&latitude={lat}&start={start}&end={end}&format=JSON"
    try:
        response = requests.get(url)
        if response.status_code == 200:
            return response.json()['properties']['parameter']
    except Exception as e:
        print(f"NASA API error for {lat},{lon}: {e}")
    return None

print(f"Starting download for {len(unique_locations)} districts. This will take ~15 minutes...")

# 2. Iterate through all districts
for index, row in unique_locations.iterrows():
    state = row['State']
    district = row['District']
    
    lat, lon = get_coordinates(state, district)
    
    if lat and lon:
        data = get_nasa_weather(lat, lon)
        if data:
            # NASA data comes as { 'YYYYMM': value }. We flatten this into rows.
            for year_month, rain_val in data['PRECTOTCORR'].items():
                temp_val = data['T2M'].get(year_month)
                
                weather_records.append({
                    'State': state,
                    'District': district,
                    'YearMonth': year_month, # e.g., '202201'
                    'Rainfall_mm': rain_val,
                    'Temperature_C': temp_val
                })
        print(f"Success: {district} ({index+1}/{len(unique_locations)})")
    else:
        print(f"Failed to find coordinates for: {district}")
        
    time.sleep(1) # Be polite to the APIs!

# 3. Save the raw weather data
weather_df = pd.DataFrame(weather_records)

Starting download for 733 districts. This will take ~15 minutes...
Success: North And Middle Andaman (1/733)
Success: Anantapur (2/733)
Success: Chittoor (3/733)
Success: East Godavari (4/733)
Success: Guntur (5/733)
Success: Krishna (6/733)
Success: Kurnool (7/733)
Success: Prakasam (8/733)
Success: Spsr Nellore (9/733)
Success: Srikakulam (10/733)
Failed to find coordinates for: Visakhapatanam
Success: Vizianagaram (12/733)
Success: West Godavari (13/733)
Success: Y.S.R. (14/733)
Success: Anjaw (15/733)
Success: Changlang (16/733)
Success: Dibang Valley (17/733)
Success: East Kameng (18/733)
Success: East Siang (19/733)
Success: Lohit (20/733)
Success: Longding (21/733)
Success: Lower Dibang Valley (22/733)
Success: Lower Subansiri (23/733)
Success: Papum Pare (24/733)
Success: Tawang (25/733)
Success: Tirap (26/733)
Success: Upper Siang (27/733)
Success: Upper Subansiri (28/733)
Success: West Kameng (29/733)
Success: West Siang (30/733)
Success: Baksa (31/733)
Success: Barpeta (32/7

In [11]:
os.makedirs('../data/raw', exist_ok=True)
weather_df.to_csv('../data/raw/nasa_monthly_weather_2014_2025.csv', index=False)

print("\nFinished! Weather data saved to data/raw/nasa_monthly_weather_2022_2025.csv")


Finished! Weather data saved to data/raw/nasa_monthly_weather_2022_2025.csv
